In [ ]:
import cv2
import matplotlib.pyplot as plt

# 画像を読み込み
img = cv2.imread('/content/drive/My Drive/Colab Notebooks/data/cv/shiba_inu/shiba_inu_34.jpg')

# 画像のサイズの確認
print(img.shape)

# 画像ファイルの表示
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.show()

In [ ]:
# 画像のピクセル値を確認
print(img)

In [ ]:
# 画像の大きさを固定する
img = cv2.resize(img, (256, 256))

# 画像のサイズの確認
print(img.shape)

# 画像ファイルの表示
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.show()

In [ ]:
import numpy as np

# RGBヒストグラムの作成
b, g, r = img[:,:,0], img[:,:,1], img[:,:,2]
hist_b, bins = np.histogram(b.ravel(), 256, [0,256])
hist_r, bins = np.histogram(r.ravel(), 256, [0,256])
hist_g, bins = np.histogram(g.ravel(), 256, [0,256])

print(hist_b)
print(hist_g)
print(hist_r)

In [ ]:
# RGBヒストグラムの描画
plt.xlim(0, 255)
plt.plot(hist_r, '-r', label='red')
plt.plot(hist_g, '-g', label='green')
plt.plot(hist_b, '-b', label='blue')
plt.xlabel('pixel value')
plt.ylabel('number of pixels')
plt.legend()
plt.show()

### 柴犬の特徴量とラベル作成

In [ ]:
import os
import cv2
import numpy as np

# ファイルの取得
files = os.listdir('/content/drive/My Drive/Colab Notebooks/data/cv/shiba_inu/')

pixels_shiba = []

for f in files:
  # 画像をカラーで読み込み
  img = cv2.imread('/content/drive/My Drive/Colab Notebooks/data/cv/shiba_inu/' + f)

  # 画像をリサイズ
  img = cv2.resize(img, (256, 256))
  # RGBヒストグラムの作成
  b, g, r = img[:,:,0], img[:,:,1], img[:,:,2]
  hist_b, bins = np.histogram(b.ravel(), 256, [0,256])
  hist_r, bins = np.histogram(r.ravel(), 256, [0,256])
  hist_g, bins = np.histogram(g.ravel(), 256, [0,256])
  tmp = hist_b.tolist() + hist_g.tolist() + hist_r.tolist()
  pixels_shiba.append(tmp)

# ラベル作成
labels_shiba = [0] * len(pixels_shiba)
labels_shiba = np.array(labels_shiba)

In [ ]:
import pandas as pd

# ピクセル値をデータフレーム形式へ変換
pixels_shiba = pd.DataFrame(pixels_shiba)

print(pixels_shiba.shape)
pixels_shiba.head()

### サモエド犬の特徴量とラベル作成

In [ ]:
# ファイルの取得
files = os.listdir('/content/drive/My Drive/Colab Notebooks/data/cv/samoyed/')

pixels_samo = []

for f in files:
  # 画像を読み込み
  img = cv2.imread('/content/drive/My Drive/Colab Notebooks/data/cv/samoyed/' + f)

  # 画像をリサイズ
  img = cv2.resize(img, (256, 256))
  # RGBヒストグラムの作成
  b, g, r = img[:,:,0], img[:,:,1], img[:,:,2]
  hist_b, bins = np.histogram(b.ravel(), 256, [0,256])
  hist_r, bins = np.histogram(r.ravel(), 256, [0,256])
  hist_g, bins = np.histogram(g.ravel(), 256, [0,256])
  tmp = hist_b.tolist() + hist_g.tolist() + hist_r.tolist()
  pixels_samo.append(tmp)

# ラベル作成
labels_samo = [1] * len(pixels_samo)
labels_samo = np.array(labels_samo)

In [ ]:
# ピクセル値をデータフレーム形式へ変換
pixels_samo = pd.DataFrame(pixels_samo)

print(pixels_samo.shape)
pixels_samo.head()

### 特徴量とラベルセットを結合

In [ ]:
from sklearn import model_selection

# 柴犬とサモエド犬の特徴量セットを結合
pixels_set = pd.concat([pixels_shiba, pixels_samo])
pixels_set = np.array(pixels_set)

# 柴犬とサモエド犬のラベルセットを結合
labels_set = np.concatenate([labels_shiba, labels_samo])

# データセットを学習と評価用に分割
trainX, testX, trainY, testY = model_selection.train_test_split(pixels_set, labels_set, test_size=0.2)

print(trainX.shape, trainY.shape) # 学習データのサイズ
print(testX.shape, testY.shape)   # 評価データのサイズ

### 交差検証法・SVMによる学習

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn import svm
from sklearn.metrics import accuracy_score

# 特徴量セットを標準化
sc = StandardScaler()
sc.fit(trainX)
trainX = sc.transform(trainX)

# K-Fold交差検定
kf = KFold(n_splits=5, shuffle=True)
# モデル精度を格納する準備
scores = []
# データをシャッフルし、訓練データとテストデータに分割
for train_id, test_id in kf.split(trainX):
    # 訓練データを使ってモデルを作成
    x = trainX[train_id]
    y = trainY[train_id]
    clf = svm.SVC()
    clf.fit(x,y)
    # テストデータにモデルを適用
    pred_y = clf.predict(trainX[test_id])
    # モデル精度を計算して格納
    score = accuracy_score(trainY[test_id], pred_y)
    scores.append(score)

# モデルの平均精度、標準偏差を確認
scores = np.array(scores)
print(scores.mean(), scores.std())

### 評価画像セットの分類

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# 特徴量セットを標準化
sc = StandardScaler()
sc.fit(testX)
testX = sc.transform(testX)

# 評価データにモデルを適用
pred = clf.predict(testX)
# 評価データの精度を計算
score = accuracy_score(testY, pred)
print(score)

# 混同行列の描画
ConfusionMatrixDisplay.from_predictions(testY, pred)
plt.show()

### 新規画像を分類する

In [ ]:
# 新規画像ファイルを指定
files = ['shiba_inu_122.jpg', 'shiba_inu_139.jpg', 'shiba_inu_153.jpg',
         'samoyed_137.jpg', 'samoyed_145.jpg']

# RGBヒストグラムを格納するリスト
pixels_new = []

# 画像ファイルを1枚ずつ読み込みRGBヒストグラム作成
for f in files:
  # 画像を読み込み
  img = cv2.imread('/content/drive/My Drive/Colab Notebooks/data/cv/wanwan/' + f)

  # 画像をリサイズ
  img = cv2.resize(img, (256, 256))
  # RGBヒストグラムの作成
  b, g, r = img[:,:,0], img[:,:,1], img[:,:,2]
  hist_b, bins = np.histogram(b.ravel(), 256, [0,256])
  hist_r, bins = np.histogram(r.ravel(), 256, [0,256])
  hist_g, bins = np.histogram(g.ravel(), 256, [0,256])
  tmp = hist_b.tolist() + hist_g.tolist() + hist_r.tolist()
  pixels_new.append(tmp)

# RGBヒストグラムをデータフレーム形式へ変換
pixels_new = pd.DataFrame(pixels_new)

print(pixels_new.shape)
pixels_new.head()

In [ ]:
# 特徴量セット（RGBヒストグラム）を標準化
sc = StandardScaler()
sc.fit(pixels_new)
pixels_new = sc.transform(pixels_new)

# 新規画像にモデルを適用して予測
pred = clf.predict(pixels_new)
print(pred)    # 予測結果が0なら柴犬、1ならサモエド犬